# 03 — Baseline Model: TF-IDF + Logistic Regression
**Goal:** Establish a baseline accuracy before fine-tuning transformers  
**Why baseline?** Provides a benchmark to measure improvement from DistilBERT/ClinicalBERT  
**Pipeline:** Raw Text → TF-IDF Vectorizer → Logistic Regression → Prediction

In [ ]:
# Imports

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Cleaned Data

df = pd.read_csv('mtsamples_cleaned.csv')
df = df.dropna(subset=['transcription'])

with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

print("Data loaded:", df.shape)
print(df['medical_specialty'].value_counts())

In [ ]:
# Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    df['transcription'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# Build Pipeline

# TF-IDF: converts text to numbers based on word importance
# ngram_range=(1,2): captures single words AND word pairs
# class_weight='balanced': handles class imbalance

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        stop_words='english',
        ngram_range=(1, 2)
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ))
])

In [ ]:
# Train

print("Training baseline model...")

pipeline.fit(X_train, y_train)

print("Training complete!")

In [ ]:
# Evaluate

y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Baseline Accuracy: {accuracy*100:.2f}%")

print("\nClassification Report:")

print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0
))

In [ ]:
# Confusion Matrix

fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=le.classes_,
    xticks_rotation=45,
    ax=ax,
    cmap='Blues'
)

plt.title('Baseline Model — Confusion Matrix')

plt.tight_layout()

plt.savefig('baseline_confusion_matrix.png', dpi=150)

plt.show()

In [ ]:
# Save Results
report = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0
)

with open('results/baseline_report.txt', 'w') as f:
    f.write("Baseline Model — TF-IDF + Logistic Regression\n")
    f.write("=" * 50 + "\n")
    f.write(f"Accuracy: {accuracy*100:.2f}%\n\n")
    f.write(report)

print(f"Baseline accuracy saved: {accuracy*100:.2f}%")
print("Next step: 04_clinicalbert_model.ipynb")